In [13]:
import numpy as np, json, os, random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import LeakyReLU, BatchNormalization
from sklearn.metrics import classification_report, f1_score
from tensorflow.keras import optimizers

In [ ]:
data = np.load("preprocessed_arrays.npz")
X_train_pad, y_train_cat = data["X_train_pad"], data["y_train_cat"]
X_val_pad,   y_val_cat   = data["X_val_pad"],   data["y_val_cat"]

with open("label_encoder_classes.json","r",encoding="utf-8") as f:
    classes = json.load(f)

with open("tokenizer.json","r",encoding="utf-8") as f:
    tok = tokenizer_from_json(f.read())

In [15]:
MAX_LEN   = X_train_pad.shape[1]
MAX_VOCAB = tok.num_words or (len(tok.word_index) + 1)

In [16]:
model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=128, input_length=MAX_LEN, mask_zero=True),
    LSTM(128, dropout=0.5, recurrent_dropout=0.3),
    Dropout(0.5),
    Dense(64),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.5),
    Dense(len(classes), activation="softmax")
])

opt = optimizers.Adam(learning_rate=5e-4)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])
model.summary()
lstm_model = model

c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\.venv\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
cb = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)
]

history = lstm_model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=20,
    batch_size=128,
    callbacks=cb,
    verbose=1
)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 69s 336ms/step - accuracy: 0.5740 - loss: 0.9042 - val_accuracy: 0.6868 - val_loss: 1.0105 - learning_rate: 5.0000e-04
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 59s 313ms/step - accuracy: 0.7359 - loss: 0.6567 - val_accuracy: 0.7238 - val_loss: 0.8236 - learning_rate: 5.0000e-04
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 70s 372ms/step - accuracy: 0.7918 - loss: 0.5270 - val_accuracy: 0.7582 - val_loss: 0.6187 - learning_rate: 5.0000e-04
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 65s 343ms/step - accuracy: 0.8291 - loss: 0.4440 - val_accuracy: 0.7695 - val_loss: 0.5989 - learning_rate: 5.0000e-04
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 83s 443ms/step - accuracy: 0.8526 - loss: 0.3895 - val_accuracy: 0.7710 - val_loss: 0.6196 - learning_rate: 5.0000e-04
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 508ms/step - accuracy: 0.8780 - loss: 0.3349
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
188/188 ━━━━━━━━━━━━━━━━━━━━ 101s 537ms/s

In [18]:
y_true = y_val_cat.argmax(axis=1)
y_pred = lstm_model.predict(X_val_pad, verbose=0).argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=classes))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

              precision    recall  f1-score   support

     negatif       0.77      0.73      0.75      2000
        notr       0.77      0.76      0.76      2000
     pozitif       0.77      0.83      0.80      2000

    accuracy                           0.77      6000
   macro avg       0.77      0.77      0.77      6000
weighted avg       0.77      0.77      0.77      6000

Macro F1: 0.768914864716037


In [19]:
def predict_text(text, model=lstm_model, tok=tok, classes=None, maxlen=100):
    seq  = tok.texts_to_sequences([text])
    pad  = pad_sequences(seq, maxlen=maxlen, padding="post", truncating="post")
    probs = model.predict(pad, verbose=0)[0]
    cls_idx = int(probs.argmax())
    return classes[cls_idx], probs

examples = [
    "Ürün gerçekten harika, çok memnun kaldım.",
    "Kargo çok geç geldi ve paket yırtıktı.",
    "Ürün fena değil, idare eder.",
    "Telefonun özellikleri iyi ama bataryası çabuk bitiyor.",
    "Ürün güzel ama fiyatına göre performansı düşük.",
    "Kargo biraz geç geldi ama satıcı çok yardımcı oldu.",
    "Sevdim ürünü, tekrar alırım.",
    "Tavsiye ederim.",
    "Berbat"
]

for ex in examples:
    label, probs = predict_text(ex, model=model, tok=tok, classes=classes, maxlen=100)
    print(f"Metin: {ex}")
    print(f"Tahmin: {label}, Olasılıklar: {probs}\n")

Metin: Ürün gerçekten harika, çok memnun kaldım.
Tahmin: pozitif, Olasılıklar: [0.00281829 0.08650701 0.9106747 ]

Metin: Kargo çok geç geldi ve paket yırtıktı.
Tahmin: notr, Olasılıklar: [0.17939673 0.80075294 0.01985032]

Metin: Ürün fena değil, idare eder.
Tahmin: notr, Olasılıklar: [0.02202464 0.976654   0.00132142]

Metin: Telefonun özellikleri iyi ama bataryası çabuk bitiyor.
Tahmin: notr, Olasılıklar: [0.14907053 0.8425225  0.00840699]

Metin: Ürün güzel ama fiyatına göre performansı düşük.
Tahmin: notr, Olasılıklar: [0.03027664 0.9385198  0.03120364]

Metin: Kargo biraz geç geldi ama satıcı çok yardımcı oldu.
Tahmin: notr, Olasılıklar: [0.1309946  0.8613312  0.00767423]

Metin: Sevdim ürünü, tekrar alırım.
Tahmin: pozitif, Olasılıklar: [0.05188563 0.425066   0.5230484 ]

Metin: Tavsiye ederim.
Tahmin: pozitif, Olasılıklar: [0.10845029 0.31469426 0.57685536]

Metin: Berbat
Tahmin: negatif, Olasılıklar: [0.8911773  0.08042195 0.02840076]



In [20]:
from pathlib import Path

current_dir = Path.cwd() / "Models"
save_path = current_dir / "lstm_model.keras"

model.save(save_path)
print(f"LSTM modeli kaydedildi: {save_path}")


LSTM modeli kaydedildi: c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\Models\lstm_model.keras
